# Pick-place-v3 PPO Split Results Visualization

This notebook visualizes the results from `pick-place_v3_ppo_split_runs`.

It uses:
```text
pick-place_v3_ppo_split_runs/results/pick-place_v3_aggregate_results.csv
pick-place_v3_ppo_split_runs/results/pick-place_v3_per_episode_results.csv
```

Run it from the folder that contains `pick-place_v3_ppo_split_runs`, or change `ROOT` below.

In [ ]:
from pathlib import Path
import pandas as pd # type: ignore
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore

ROOT = Path('pick-place_v3_ppo_split_runs')
RESULTS = ROOT / 'results'
agg_path = RESULTS / 'pick-place_v3_aggregate_results.csv'
ep_path = RESULTS / 'pick-place_v3_per_episode_results.csv'

assert agg_path.exists(), f'Missing {agg_path}'
agg = pd.read_csv(agg_path)
episodes = pd.read_csv(ep_path) if ep_path.exists() else None

print('Aggregate shape:', agg.shape)
print('Per-episode shape:', None if episodes is None else episodes.shape)
display(agg.head())

## 1. Overview

In [ ]:
print('Configs:', sorted(agg['config_name'].unique()))
print('Splits:', sorted(agg['split_id'].unique()))
print('Split seeds:', sorted(agg['split_seed'].unique()))
print('Timesteps:', sorted(agg['total_timesteps'].unique()))
display(agg[['config_name','split_id','split_seed','train_success_rate','test_success_rate','train_avg_first_success_step','test_avg_first_success_step','success_gap']].sort_values(['config_name','split_id']))

## 2. Train/Test success rate by config and split
This shows whether each PPO configuration generalizes from the 45 train variations to the 5 held-out test variations.

In [ ]:
configs = sorted(agg['config_name'].unique())
for config in configs:
    d = agg[agg['config_name'] == config].sort_values('split_id')
    x = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(8,5))
    width = 0.38
    ax.bar(x-width/2, d['train_success_rate'], width, label='train')
    ax.bar(x+width/2, d['test_success_rate'], width, label='test')
    ax.set_title(f'Pick-place-v3 success rate by split — {config}')
    ax.set_xlabel('Split ID')
    ax.set_ylabel('Success rate')
    ax.set_ylim(0,1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(d['split_id'])
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.show()

## 3. Mean success across splits

In [ ]:
summary = (agg.groupby('config_name')
    .agg(
        train_success_mean=('train_success_rate','mean'),
        train_success_std=('train_success_rate','std'),
        test_success_mean=('test_success_rate','mean'),
        test_success_std=('test_success_rate','std'),
        train_first_success_mean=('train_avg_first_success_step','mean'),
        test_first_success_mean=('test_avg_first_success_step','mean'),
        success_gap_mean=('success_gap','mean'),
        runs=('split_id','count')
    ).reset_index())
display(summary)

x = np.arange(len(summary))
fig, ax = plt.subplots(figsize=(9,5))
width=0.38
ax.bar(x-width/2, summary['train_success_mean'], width, yerr=summary['train_success_std'].fillna(0), capsize=4, label='train')
ax.bar(x+width/2, summary['test_success_mean'], width, yerr=summary['test_success_std'].fillna(0), capsize=4, label='test')
ax.set_title('Pick-place-v3 mean success across 3 splits')
ax.set_xlabel('PPO config')
ax.set_ylabel('Mean success rate')
ax.set_ylim(0,1.05)
ax.set_xticks(x)
ax.set_xticklabels(summary['config_name'], rotation=20, ha='right')
ax.grid(True, axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4. First success step
Lower is better. NaN means the model did not reach success in those episodes.

In [ ]:
for config in configs:
    d = agg[agg['config_name'] == config].sort_values('split_id')
    x = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(8,5))
    width = 0.38
    ax.bar(x-width/2, d['train_avg_first_success_step'], width, label='train')
    ax.bar(x+width/2, d['test_avg_first_success_step'], width, label='test')
    ax.set_title(f'Pick-place-v3 first success step — {config}')
    ax.set_xlabel('Split ID')
    ax.set_ylabel('Average first success step')
    ax.set_xticks(x)
    ax.set_xticklabels(d['split_id'])
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.show()

## 5. Per-episode distributions
This is useful to see instability and failures.

In [ ]:
if episodes is not None:
    display(episodes.head())
    print("Columns:")
    print(episodes.columns.tolist())

    # Find likely column names
    success_col = "success" if "success" in episodes.columns else None

    # group may be called group, split_group, eval_group, dataset, etc.
    possible_group_cols = ["group", "eval_group", "split_group", "dataset", "phase"]
    group_col = next((c for c in possible_group_cols if c in episodes.columns), None)

    # config may be called config, config_name, model, run_config, etc.
    possible_config_cols = ["config_name", "config", "run_config", "model_config"]
    config_col = next((c for c in possible_config_cols if c in episodes.columns), None)

    # split id may or may not exist
    possible_split_cols = ["split_id", "split", "split_seed"]
    split_col = next((c for c in possible_split_cols if c in episodes.columns), None)

    print("\nDetected:")
    print("success_col:", success_col)
    print("group_col:", group_col)
    print("config_col:", config_col)
    print("split_col:", split_col)

    if success_col is not None:
        groupby_cols = []

        if config_col is not None:
            groupby_cols.append(config_col)

        if split_col is not None:
            groupby_cols.append(split_col)

        if group_col is not None:
            groupby_cols.append(group_col)

        if len(groupby_cols) > 0:
            ep_sum = (
                episodes
                .groupby(groupby_cols)[success_col]
                .mean()
                .reset_index()
                .rename(columns={success_col: "success_rate"})
            )
            display(ep_sum.head(30))
        else:
            print("No grouping columns found. Overall success:")
            print(episodes[success_col].mean())
    else:
        print("Could not find a success column.")

else:
    print("No per-episode CSV found.")

## 6. Save presentation figures

In [ ]:
fig_dir = ROOT / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# summary plot
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(summary))
width=0.38
ax.bar(x-width/2, summary['train_success_mean'], width, yerr=summary['train_success_std'].fillna(0), capsize=4, label='train')
ax.bar(x+width/2, summary['test_success_mean'], width, yerr=summary['test_success_std'].fillna(0), capsize=4, label='test')
ax.set_title('Pick-place-v3 mean success across 3 splits')
ax.set_xlabel('PPO config')
ax.set_ylabel('Mean success rate')
ax.set_ylim(0,1.05)
ax.set_xticks(x)
ax.set_xticklabels(summary['config_name'], rotation=20, ha='right')
ax.grid(True, axis='y', alpha=0.3)
ax.legend()
plt.tight_layout()
out = fig_dir / 'pick_place_mean_success_by_config.png'
fig.savefig(out, dpi=200, bbox_inches='tight')
plt.close(fig)
print('Saved:', out)

## 7. Suggested interpretation

- `careful_pick` is the strongest and most stable configuration.
- `light_entropy_pick` also performs well and reaches 100% test success on all three splits, but train success is slightly lower.
- `base_pick` is weaker and more split-sensitive, especially on split 1.
- `short_rollout_pick` fails badly, suggesting that the shorter rollout of 512 steps is not enough for this task/setup.
- `pick-place-v3` is harder than `button-press-v3`, but easier/more stable than the problematic basketball runs.